In [1]:
import math
import scipy.special as ss
from fractions import Fraction
import os
import numpy as np
from collections import Counter
import sys

try:
    import gf2matrix
    print("Successfully imported gf2matrix.py")
except ImportError:
    print("ERROR: Could not import 'gf2matrix'.")
    print("Please make sure the 'gf2matrix.py' file is in the same directory as this script.")
    sys.exit()

# --- Cell 1: Frequency (Monobit) Test ---
def frequency_test(input_str, n):
    """
    Performs a frequency test on a binary string.
    """
    ones = input_str.count('1')
    zeroes = input_str.count('0')
    s = abs(ones - zeroes)
    p = math.erfc(float(s)/(math.sqrt(float(n)) * math.sqrt(2.0)))
    success = (p >= 0.01)
    return [zeroes, ones, s, p, success]

# --- Cell 2: Block Frequency Test ---
def block_frequency_test(input_str, n, M=128):
    """
    Performs the Block Frequency Test on a binary string.
    """
    num_blocks = math.floor(n / M)
    block_size = M
    
    if n < 100 or num_blocks < 1:
        return [0.0, 0.0, False]

    proportions = []
    for i in range(num_blocks):
        block = input_str[i * block_size : (i + 1) * block_size]
        ones = block.count('1')
        proportions.append(Fraction(ones, block_size))

    chisq = 0.0
    for prop in proportions:
        chisq += 4.0 * block_size * ((prop - Fraction(1, 2))**2)
    
    p_value = ss.gammaincc(num_blocks / 2.0, float(chisq) / 2.0)
    success = (p_value >= 0.01)
    return [float(chisq), p_value, success]

# --- Cell 3: Runs Test ---
def runs_test(input_str, n):
    """
    Performs the NIST Runs Test on a binary string.
    """
    ones = input_str.count('1')
    zeroes = input_str.count('0')
    
    prop = float(ones) / float(n)
    tau = 2.0 / math.sqrt(n)
    if abs(prop - 0.5) >= tau:
        return [zeroes, ones, prop, 0.0, 0.0, False]

    vobs = 1.0
    for i in range(n - 1):
        if input_str[i] != input_str[i+1]:
            vobs += 1.0

    p_value = math.erfc(abs(vobs - (2.0 * n * prop * (1.0 - prop))) / 
                       (2.0 * math.sqrt(2.0 * n) * prop * (1.0 - prop)))
    success = (p_value >= 0.01)
    return [zeroes, ones, prop, vobs, p_value, success]

# --- Cell 4: Longest Run of Ones Test ---
def longest_run_test(input_str):
    """
    Performs the NIST Longest Run of Ones in a Block Test. (M=8)
    """
    n = len(input_str)
    M = 8   # Block size
    K = 3   # Degrees of freedom
    PI = [0.2148, 0.3672, 0.2305, 0.1875] 
    N = math.floor(n / M)
    
    if N < 16:
        return [0.0, 0.0, False]

    v = [0, 0, 0, 0]
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        current_run = 0
        longest_run = 0
        for bit in block:
            if bit == '1':
                current_run += 1
                if current_run > longest_run:
                    longest_run = current_run
            else:
                current_run = 0
        
        if longest_run <= 1: v[0] += 1
        elif longest_run == 2: v[1] += 1
        elif longest_run == 3: v[2] += 1
        else: v[3] += 1
    
    chi_sq = 0.0
    for i in range(K + 1):
        numerator = (v[i] - N * PI[i])**2
        denominator = N * PI[i]
        chi_sq += numerator / denominator
        
    p_value = ss.gammaincc(K / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [chi_sq, p_value, success]

# --- Cell 5: Binary Matrix Rank Test ---
def binary_matrix_rank_test(input_str, n, M=32, Q=32):
    """
    Performs the NIST Binary Matrix Rank Test. Requires gf2matrix.py
    """
    num_blocks = int(math.floor(n / (M * Q)))
    if num_blocks < 38:
        return [0.0, 0.0, False]

    product = 1.0
    for i in range(M):
        product *= (1.0 - 2.0**(i - Q)) * (1.0 - 2.0**(i - M)) / (1.0 - 2.0**(i - M))
    p_full_rank = product * (2.0**(M * (Q + M - M) - (M * Q)))

    product = 1.0
    for i in range(M - 1):
        product *= (1.0 - 2.0**(i - Q)) * (1.0 - 2.0**(i - M)) / (1.0 - 2.0**(i - (M-1)))
    p_rank_m1 = product * (2.0**((M-1)*(Q+M-(M-1)) - M*Q))
    
    p_remainder = 1.0 - p_full_rank - p_rank_m1
    fm_count = 0; fmm_count = 0; rem_count = 0

    for blk_num in range(num_blocks):
        block_str = input_str[blk_num*M*Q : (blk_num+1)*M*Q]
        block = [int(bit) for bit in block_str]
        
        matrix = gf2matrix.matrix_from_bits(M, Q, block, blk_num)
        rank = gf2matrix.rank(M, Q, matrix, blk_num)

        if rank == M: fm_count += 1
        elif rank == M - 1: fmm_count += 1
        else: rem_count += 1
            
    chisq = (((fm_count - p_full_rank * num_blocks)**2) / (p_full_rank * num_blocks) +
             ((fmm_count - p_rank_m1 * num_blocks)**2) / (p_rank_m1 * num_blocks) +
             ((rem_count - p_remainder * num_blocks)**2) / (p_remainder * num_blocks))
    
    p_value = math.exp(-chisq / 2.0)
    success = (p_value >= 0.01)
    return [chisq, p_value, success]

# --- Cell 6: Spectral (DFT) Test ---
def spectral_test(input_str, n):
    """
    Performs the NIST Discrete Fourier Transform (DFT) / Spectral Test.
    """
    ts = [(1 if bit == '1' else -1) for bit in input_str]
    ts_np = np.array(ts)
    fs = np.fft.fft(ts_np)
    mags = abs(fs)[:n//2]
    T = math.sqrt(math.log(1.0 / 0.05) * n)
    N0 = 0.95 * (n / 2.0)
    N1 = float(np.sum(mags < T))
    d = (N1 - N0) / math.sqrt((n * 0.95 * 0.05) / 4.0)
    p_value = math.erfc(abs(d) / math.sqrt(2))
    success = (p_value >= 0.01)
    return [N0, N1, d, p_value, success]

# --- Cell 7: Non-Overlapping Template Test ---
def non_overlapping_template_test(input_str):
    """
    Performs the NIST Non-Overlapping Template Matching Test.
    """
    n = len(input_str)
    templates_int = [ [0, 1], [1, 0], [0, 0, 1], [0, 1, 1], [1, 0, 0], [1, 1, 0],
                      [0, 0, 0, 1], [0, 0, 1, 1], [0, 1, 1, 1], [1, 0, 0, 0], [1, 1, 0, 0], [1, 1, 1, 0] ]
    B_int = templates_int[0] # Using a fixed template '01' for consistency
    m = len(B_int)
    template_str = "".join(map(str, B_int))
    N = 8    # The test is run on N blocks
    M = n // N # Length of each block
    
    if M < 21:
        return [0.0, 0.0, 0.0, 0.0, False]

    W = [] 
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        count = 0
        position = 0
        while position < (M - m + 1):
            if block[position : position + m] == template_str:
                count += 1
                position += m
            else:
                position += 1
        W.append(count)

    mu = (M - m + 1) / (2**m)
    sigma_sq = M * ((1.0 / (2**m)) - ((2.0 * m - 1.0) / (2**(2 * m))))
    chi_sq = 0.0
    for count in W:
        chi_sq += ((count - mu)**2) / sigma_sq
        
    p_value = ss.gammaincc(N / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [mu, sigma_sq, chi_sq, p_value, success, template_str]

# --- Cell 8: Overlapping Template Test (and helpers) ---
def lgamma(x):
    return math.log(ss.gamma(x))

def Pr(u, eta):
    if u == 0:
        return math.exp(-eta)
    else:
        sum_val = 0.0
        for l in range(1, u + 1):
            sum_val += math.exp(-eta - u * math.log(2) + l * math.log(eta) - lgamma(l + 1) + lgamma(u) - lgamma(l) - lgamma(u - l + 1))
        return sum_val

def overlapping_template_test(input_str):
    """
    Performs the NIST Overlapping Template Matching Test.
    """
    n = len(input_str)
    m = 10   # Length of the template pattern
    N = 968  # Number of blocks
    M = 1032 # Length of each block
    K = 5    # Degrees of freedom
    template_str = '1' * m
    
    if n < (M * N):
        return [0.0, 0.0, False, [0]*6]

    v = [0] * (K + 1)
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        count = 0
        for j in range(M - m + 1):
            if block[j : j + m] == template_str:
                count += 1
        
        if count >= K:
            v[K] += 1
        else:
            v[count] += 1

    lambd = (M - m + 1.0) / (2.0**m)
    eta = lambd / 2.0
    pi = [Pr(i, eta) for i in range(K)]
    pi.append(1.0 - sum(pi)) 

    chisq = 0.0
    for i in range(K + 1):
        chisq += ((v[i] - N * pi[i])**2) / (N * pi[i])
        
    p_value = ss.gammaincc((K / 2.0), chisq / 2.0)
    success = (p_value >= 0.01)
    return [chisq, p_value, success, v]

# --- Cell 9: Universal Test ---
def universal_test(input_str):
    """
    Performs Maurer's Universal Statistical Test.
    """
    n = len(input_str)
    if n >= 1059061760: L = 16
    elif n >= 496435200:  L = 15
    elif n >= 231669760:  L = 14
    elif n >= 107560960:  L = 13
    elif n >= 496435200:  L = 12
    elif n >= 22753280:   L = 11
    elif n >= 10342400:   L = 10
    elif n >= 4654080:    L = 9
    elif n >= 2068480:    L = 8
    elif n >= 904960:     L = 7
    elif n >= 387840:     L = 6
    else:
        return [0] * 3 

    num_blocks = math.floor(n / L)
    Q = 10 * (2**L)
    K = num_blocks - Q
    
    if K <= 0:
        return [0] * 3

    num_symbols = 2**L
    T = [0] * num_symbols
    for i in range(Q):
        pattern = input_str[i * L : (i + 1) * L]
        idx = int(pattern, 2)
        T[idx] = i + 1

    log_sum = 0.0
    for i in range(Q, num_blocks):
        pattern = input_str[i * L : (i + 1) * L]
        j = int(pattern, 2)
        distance = i + 1 - T[j]
        T[j] = i + 1
        log_sum += math.log2(distance)

    fn = log_sum / K
    ev_table = [0, 0.73264948, 1.5374383, 2.40160681, 3.31122472,
                4.25342659, 5.2177052, 6.1962507, 7.1836656,
                8.1764248, 9.1723243, 10.170032, 11.168765,
                12.168070, 13.167693, 14.167488, 15.167379]
    var_table = [0, 0.690, 1.338, 1.901, 2.358, 2.705, 2.954, 3.125,
                 3.238, 3.311, 3.356, 3.384, 3.401, 3.410, 3.416,
                 3.419, 3.421]
                 
    expected_value = ev_table[L]
    variance = var_table[L]
    
    arg = abs(fn - expected_value) / (math.sqrt(2 * variance))
    p_value = math.erfc(arg)
    success = (p_value >= 0.01)
    return [fn, p_value, success]

# --- Cell 10: Linear Complexity Test (and helpers) ---
def padding(input_str, n):
    while len(input_str) < n:
        input_str = '0' + input_str
    return input_str

def berlekamp_massey(input_str):
    n = len(input_str)
    b = '1' + '0' * (n - 1)
    c = '1' + '0' * (n - 1)
    L = 0
    m = -1
    N = 0
    while N < n:
        d = int(input_str[N], 2)
        if L > 0:
            k_str = c[1 : L + 1]
            h_str = input_str[N - L : N][::-1]
            k_int = int(k_str, 2)
            h_int = int(h_str, 2)
            r = bin(k_int & h_int)[2:].count('1')
            d = d ^ (r % 2)

        if d != 0:
            t = c
            k_str = c[N - m : n]
            k_int = int(k_str, 2)
            h_str = b[0 : n - N + m]
            h_int = int(h_str, 2)
            k_int = k_int ^ h_int
            c = c[0 : N - m] + padding(bin(k_int)[2:], n - N + m)
            if L <= (N / 2):
                L = N + 1 - L
                m = N
                b = t
        N += 1
    return L, c[0:L]

def linear_complexity_test(input_str, M=512):
    n = len(input_str)
    K = 6  # Degrees of freedom
    N = math.floor(n / M)
    if n < 1000000:
        return [0.0, 0.0, False, [0]*7]

    LC = [berlekamp_massey(input_str[i * M : (i + 1) * M])[0] for i in range(N)]
    mu = (M / 2.0) + ((((-1)**(M + 1)) + 9.0) / 36.0) - (((M / 3.0) + (2.0 / 9.0)) / (2**M))
    T = [(((-1.0)**M) * (lc - mu) + (2.0 / 9.0)) for lc in LC]
    
    v = [0] * (K + 1)
    for t in T:
        if   t <= -2.5: v[0] += 1
        elif t <= -1.5: v[1] += 1
        elif t <= -0.5: v[2] += 1
        elif t <= 0.5:  v[3] += 1
        elif t <= 1.5:  v[4] += 1
        elif t <= 2.5:  v[5] += 1
        else:           v[6] += 1

    pi = [0.010417, 0.03125, 0.125, 0.5, 0.25, 0.0625, 0.020833]
    chi_sq = sum(((v[i] - N * pi[i])**2.0) / (N * pi[i]) for i in range(K + 1))
    p_value = ss.gammaincc(K / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [chi_sq, p_value, success, v]

# --- Cell 11: Serial Test (and helpers) ---
def int_to_pattern_str(n, m):
    return bin(n)[2:].zfill(m)

def psi_sq(m, n, padded_input):
    if m == 0:
        return 0.0
    counts = [0] * (2**m)
    for i in range(n):
        pattern_str = padded_input[i : i + m]
        idx = int(pattern_str, 2)
        counts[idx] += 1
    
    psi_sq_m = 0.0
    for count in counts:
        psi_sq_m += count**2
        
    psi_sq_m = (psi_sq_m * (2**m) / n) - n
    return psi_sq_m

def serial_test(input_str, patternlen=None):
    n = len(input_str)
    if patternlen is not None:
        m = patternlen
    else:
        m = math.floor(math.log2(n)) - 2
        if m < 1: m = 1
    
    padded_input = input_str + input_str[0 : m - 1]
    
    psi_sq_m = psi_sq(m, n, padded_input)
    psi_sq_m_minus_1 = psi_sq(m - 1, n, padded_input)
    psi_sq_m_minus_2 = psi_sq(m - 2, n, padded_input)
    
    delta1 = psi_sq_m - psi_sq_m_minus_1
    delta2 = psi_sq_m - (2 * psi_sq_m_minus_1) + psi_sq_m_minus_2

    p_value1 = ss.gammaincc(2**(m - 2), delta1 / 2.0)
    p_value2 = ss.gammaincc(2**(m - 3), delta2 / 2.0)
    
    success = (p_value1 >= 0.01) and (p_value2 >= 0.01)
    return [delta1, delta2, p_value1, p_value2, success]

# --- Cell 12: Approximate Entropy Test ---
def approximate_entropy_test(input_str):
    n = len(input_str)
    m = math.floor(math.log2(n)) - 6
    if m < 2: m = 2

    phi = [0.0, 0.0]
    for j in range(m, m + 2):
        if j == 0:
            phi[j-m] = 0.0
            continue
            
        padded_input = input_str + input_str[0 : j - 1]
        counts = [0] * (2**j)
        for i in range(n):
            pattern = padded_input[i : i + j]
            idx = int(pattern, 2)
            counts[idx] += 1
            
        probabilities = [count / n for count in counts]
        
        temp_phi = 0.0
        for prob in probabilities:
            if prob > 0:
                temp_phi += prob * math.log(prob)
        phi[j-m] = temp_phi

    appen_m = phi[0] - phi[1]
    chi_sq = 2 * n * (math.log(2) - appen_m)
    p_value = ss.gammaincc(2**(m - 1), chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [appen_m, chi_sq, p_value, success]

# --- Cell 13: Cumulative Sums Test (and helpers) ---
def normcdf(n):
    return 0.5 * math.erfc(-n * math.sqrt(0.5))

def p_value(n, z):
    sum_a = 0.0
    start_k = int(math.floor((((float(-n) / z) + 1.0) / 4.0)))
    end_k = int(math.floor((((float(n) / z) - 1.0) / 4.0)))
    for k in range(start_k, end_k + 1):
        c1 = (((4.0 * k) + 1.0) * z) / math.sqrt(n)
        d1 = normcdf(c1)
        c2 = (((4.0 * k) - 1.0) * z) / math.sqrt(n)
        e1 = normcdf(c2)
        sum_a = sum_a + d1 - e1

    sum_b = 0.0
    start_k = int(math.floor((((float(-n) / z) - 3.0) / 4.0)))
    end_k = int(math.floor((((float(n) / z) - 1.0) / 4.0)))
    for k in range(start_k, end_k + 1):
        c1 = (((4.0 * k) + 3.0) * z) / math.sqrt(n)
        d1 = normcdf(c1)
        c2 = (((4.0 * k) + 1.0) * z) / math.sqrt(n)
        e1 = normcdf(c2)
        sum_b = sum_b + d1 - e1

    p = 1.0 - sum_a + sum_b
    return p

def cumulative_sums_test(input_str):
    n = len(input_str)
    x = [(int(bit) * 2 - 1) for bit in input_str]
    
    pos = 0; forward_max = 0
    for e in x:
        pos += e
        if abs(pos) > forward_max:
            forward_max = abs(pos)
    p_forward = p_value(n, forward_max)
    
    pos = 0; backward_max = 0
    for e in reversed(x):
        pos += e
        if abs(pos) > backward_max:
            backward_max = abs(pos)
    p_backward = p_value(n, backward_max)
    
    success = (p_forward >= 0.01) and (p_backward >= 0.01)
    return [p_forward, p_backward, success]

# --- Cell 14: Random Excursions Test ---
def random_excursions_test(input_str):
    n = len(input_str)
    if n < 1000000:
        return [False, None] # Not enough data

    x_np = np.array(list(input_str), dtype=np.int8) * 2 - 1
    s = np.cumsum(x_np)
    s_prime = np.concatenate(([0], s, [0]))
    zero_crossings = np.where(s_prime == 0)[0]
    cycles = [s_prime[zero_crossings[i]:zero_crossings[i+1]+1] for i in range(len(zero_crossings) - 1)]
    J = len(cycles)
    
    if J < 10:
        return [False, None] # Not enough cycles

    states = [-4, -3, -2, -1, 1, 2, 3, 4]
    pi_kx = [
        [0.5, 0.25, 0.125, 0.0625, 0.03125, 0.03125],
        [0.75, 0.0625, 0.046875, 0.03515625, 0.0263671875, 0.0791015625],
        [0.8333333333, 0.0277777778, 0.0231481481, 0.0192901235, 0.0160751029, 0.0803755123],
        [0.875, 0.015625, 0.013671875, 0.0119628906, 0.0104675293, 0.0732727051],
    ]

    results = []
    overall_success = True
    
    v_counts = {state: [0] * 6 for state in states}
    for cycle in cycles:
        cycle_counter = Counter(cycle)
        for x_state in states:
            count = cycle_counter.get(x_state, 0)
            if count >= 5: v_counts[x_state][5] += 1
            else: v_counts[x_state][count] += 1
    
    for x_state in states:
        v_xk = v_counts[x_state]
        pi = pi_kx[abs(x_state) - 1]
        chi_sq = 0.0
        for k in range(6):
            numerator = (v_xk[k] - J * pi[k])**2
            denominator = J * pi[k]
            if denominator == 0: continue
            chi_sq += numerator / denominator
            
        p_value = ss.gammaincc(5.0 / 2.0, chi_sq / 2.0)
        if p_value < 0.01: overall_success = False
        results.append({'state': x_state, 'p_value': p_value, 'pass': (p_value >= 0.01)})

    return overall_success, results

# --- Cell 16: Random Excursions Variant Test ---
def random_excursions_variant_test(input_str):
    n = len(input_str)
    if n < 1000000:
        return [False, 0, None] # Not enough data

    x_np = np.array(list(input_str), dtype=np.int8) * 2 - 1
    s = np.cumsum(x_np)
    s_prime = np.concatenate(([0], s))
    J = np.count_nonzero(s_prime == 0)

    if J < 10:
        return [False, J, None] # Not enough cycles

    states = list(range(-9, 0)) + list(range(1, 10))
    results = []
    overall_success = True
    
    state_counts = Counter(s_prime)
    
    for x_state in states:
        count_x = state_counts.get(x_state, 0)
        numerator = abs(count_x - J)
        denominator = math.sqrt(2.0 * J * (4.0 * abs(x_state) - 2.0))
        
        if denominator == 0:
            p_value = 0.0
        else:
            p_value = ss.erfc(numerator / denominator)
        
        if p_value < 0.01:
            overall_success = False
        
        results.append({'state': x_state, 'count': count_x, 'p_value': p_value, 'pass': (p_value >= 0.01)})
        
    return overall_success, J, results

# ===================================================
# MAIN EXECUTION BLOCK — DIRECT .BIN TESTING
# ===================================================
if __name__ == "__main__":

    BIN_FILE = "hybrid_csprng_output (1).bin"
    BITS_TO_TEST = 1_280_000   # >= 1e6 required for full NIST

    print(f"Reading binary data from '{BIN_FILE}'...")

    try:
        with open(BIN_FILE, "rb") as f:
            byte_data = f.read()

        byte_array = np.frombuffer(byte_data, dtype=np.uint8)
        bit_array = np.unpackbits(byte_array)

        if len(bit_array) < BITS_TO_TEST:
            raise ValueError(
                f"Not enough bits: need {BITS_TO_TEST}, got {len(bit_array)}"
            )

        bit_array = bit_array[:BITS_TO_TEST]
        binary_data = "".join(bit_array.astype(str))
        data_length = len(binary_data)

        print(f"Loaded {data_length} bits successfully.")

    except Exception as e:
        print("ERROR reading .bin file:", e)
        sys.exit(1)

    print("\nRunning all 15 NIST Statistical Tests...")
    print("=" * 70)
    print(f"{'Test Name':<30} | {'P-Value(s)':<25} | {'Result'}")
    print("-" * 70)

    all_tests_passed = True

    def report(name, passed, pvals):
        print(f"{name:<30} | {pvals:<25} | {'PASS' if passed else 'FAIL'}")
        return passed

    # 1. Frequency
    r = frequency_test(binary_data, data_length)
    report("Frequency", r[4], f"{r[3]:.6f}")

    # 2. Block Frequency
    r = block_frequency_test(binary_data, data_length)
    report("Block Frequency", r[2], f"{r[1]:.6f}")

    # 3. Runs
    r = runs_test(binary_data, data_length)
    report("Runs", r[5], f"{r[4]:.6f}")

    # 4. Longest Run of Ones
    r = longest_run_test(binary_data)
    report("Longest Run", r[2], f"{r[1]:.6f}")

    # 5. Rank
    r = binary_matrix_rank_test(binary_data, data_length)
    report("Binary Matrix Rank", r[2], f"{r[1]:.6f}")

    # 6. Spectral (DFT)
    r = spectral_test(binary_data, data_length)
    report("Spectral (DFT)", r[4], f"{r[3]:.6f}")

    # 7. Non-overlapping Template
    r = non_overlapping_template_test(binary_data)
    report("Non-overlap Template", r[4], f"{r[3]:.6f}")

    # 8. Overlapping Template
    r = overlapping_template_test(binary_data)
    report("Overlapping Template", r[2], f"{r[1]:.6f}")

    # 9. Universal
    r = universal_test(binary_data)
    report("Universal", r[2], f"{r[1]:.6f}")

    # 10. Linear Complexity
    r = linear_complexity_test(binary_data)
    report("Linear Complexity", r[2], f"{r[1]:.6f}")

    # 11. Serial
    r = serial_test(binary_data, 10)
    report("Serial", r[4], f"{r[2]:.6f}, {r[3]:.6f}")

    # 12. Approximate Entropy
    r = approximate_entropy_test(binary_data)
    report("Approximate Entropy", r[3], f"{r[2]:.6f}")

    # 13. Cumulative Sums
    r = cumulative_sums_test(binary_data)
    report("Cumulative Sums", r[2], f"{r[0]:.6f}, {r[1]:.6f}")

    # 14. Random Excursions
    passed, res = random_excursions_test(binary_data)
    if res is None:
        report("Random Excursions", False, "Insufficient cycles")
    else:
        min_p = min(x["p_value"] for x in res)
        report("Random Excursions", passed, f"{min_p:.6f}")

    # 15. Random Excursions Variant
    passed, J, res = random_excursions_variant_test(binary_data)
    if res is None:
        report("Random Excursions Var", False, "Insufficient cycles")
    else:
        min_p = min(x["p_value"] for x in res)
        report("Random Excursions Var", passed, f"{min_p:.6f}")

    print("=" * 70)
    if all_tests_passed:
        print("✅ CONGRATULATIONS: Passed all 15 NIST tests.")
    else:
        print("❌ Some tests failed. Review p-values above.")


Successfully imported gf2matrix.py
Reading binary data from 'hybrid_csprng_output (1).bin'...
Loaded 1280000 bits successfully.

Running all 15 NIST Statistical Tests...
Test Name                      | P-Value(s)                | Result
----------------------------------------------------------------------
Frequency                      | 0.655975                  | PASS
Block Frequency                | 0.997503                  | PASS
Runs                           | 0.861210                  | PASS
Longest Run                    | 0.814692                  | PASS
Binary Matrix Rank             | 0.498608                  | PASS
Spectral (DFT)                 | 0.123292                  | PASS
Non-overlap Template           | 0.894202                  | PASS
Overlapping Template           | 0.869782                  | PASS
Universal                      | 0.998407                  | PASS
Linear Complexity              | 0.746152                  | PASS
Serial                         

In [2]:
import math
import scipy.special as ss
from fractions import Fraction
import os
import numpy as np
from collections import Counter
import sys

try:
    import gf2matrix
    print("Successfully imported gf2matrix.py")
except ImportError:
    print("ERROR: Could not import 'gf2matrix'.")
    print("Please make sure the 'gf2matrix.py' file is in the same directory as this script.")
    sys.exit()

# --- Cell 1: Frequency (Monobit) Test ---
def frequency_test(input_str, n):
    """
    Performs a frequency test on a binary string.
    """
    ones = input_str.count('1')
    zeroes = input_str.count('0')
    s = abs(ones - zeroes)
    p = math.erfc(float(s)/(math.sqrt(float(n)) * math.sqrt(2.0)))
    success = (p >= 0.01)
    return [zeroes, ones, s, p, success]

# --- Cell 2: Block Frequency Test ---
def block_frequency_test(input_str, n, M=128):
    """
    Performs the Block Frequency Test on a binary string.
    """
    num_blocks = math.floor(n / M)
    block_size = M
    
    if n < 100 or num_blocks < 1:
        return [0.0, 0.0, False]

    proportions = []
    for i in range(num_blocks):
        block = input_str[i * block_size : (i + 1) * block_size]
        ones = block.count('1')
        proportions.append(Fraction(ones, block_size))

    chisq = 0.0
    for prop in proportions:
        chisq += 4.0 * block_size * ((prop - Fraction(1, 2))**2)
    
    p_value = ss.gammaincc(num_blocks / 2.0, float(chisq) / 2.0)
    success = (p_value >= 0.01)
    return [float(chisq), p_value, success]

# --- Cell 3: Runs Test ---
def runs_test(input_str, n):
    """
    Performs the NIST Runs Test on a binary string.
    """
    ones = input_str.count('1')
    zeroes = input_str.count('0')
    
    prop = float(ones) / float(n)
    tau = 2.0 / math.sqrt(n)
    if abs(prop - 0.5) >= tau:
        return [zeroes, ones, prop, 0.0, 0.0, False]

    vobs = 1.0
    for i in range(n - 1):
        if input_str[i] != input_str[i+1]:
            vobs += 1.0

    p_value = math.erfc(abs(vobs - (2.0 * n * prop * (1.0 - prop))) / 
                       (2.0 * math.sqrt(2.0 * n) * prop * (1.0 - prop)))
    success = (p_value >= 0.01)
    return [zeroes, ones, prop, vobs, p_value, success]

# --- Cell 4: Longest Run of Ones Test ---
def longest_run_test(input_str):
    """
    Performs the NIST Longest Run of Ones in a Block Test. (M=8)
    """
    n = len(input_str)
    M = 8   # Block size
    K = 3   # Degrees of freedom
    PI = [0.2148, 0.3672, 0.2305, 0.1875] 
    N = math.floor(n / M)
    
    if N < 16:
        return [0.0, 0.0, False]

    v = [0, 0, 0, 0]
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        current_run = 0
        longest_run = 0
        for bit in block:
            if bit == '1':
                current_run += 1
                if current_run > longest_run:
                    longest_run = current_run
            else:
                current_run = 0
        
        if longest_run <= 1: v[0] += 1
        elif longest_run == 2: v[1] += 1
        elif longest_run == 3: v[2] += 1
        else: v[3] += 1
    
    chi_sq = 0.0
    for i in range(K + 1):
        numerator = (v[i] - N * PI[i])**2
        denominator = N * PI[i]
        chi_sq += numerator / denominator
        
    p_value = ss.gammaincc(K / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [chi_sq, p_value, success]

# --- Cell 5: Binary Matrix Rank Test ---
def binary_matrix_rank_test(input_str, n, M=32, Q=32):
    """
    Performs the NIST Binary Matrix Rank Test. Requires gf2matrix.py
    """
    num_blocks = int(math.floor(n / (M * Q)))
    if num_blocks < 38:
        return [0.0, 0.0, False]

    product = 1.0
    for i in range(M):
        product *= (1.0 - 2.0**(i - Q)) * (1.0 - 2.0**(i - M)) / (1.0 - 2.0**(i - M))
    p_full_rank = product * (2.0**(M * (Q + M - M) - (M * Q)))

    product = 1.0
    for i in range(M - 1):
        product *= (1.0 - 2.0**(i - Q)) * (1.0 - 2.0**(i - M)) / (1.0 - 2.0**(i - (M-1)))
    p_rank_m1 = product * (2.0**((M-1)*(Q+M-(M-1)) - M*Q))
    
    p_remainder = 1.0 - p_full_rank - p_rank_m1
    fm_count = 0; fmm_count = 0; rem_count = 0

    for blk_num in range(num_blocks):
        block_str = input_str[blk_num*M*Q : (blk_num+1)*M*Q]
        block = [int(bit) for bit in block_str]
        
        matrix = gf2matrix.matrix_from_bits(M, Q, block, blk_num)
        rank = gf2matrix.rank(M, Q, matrix, blk_num)

        if rank == M: fm_count += 1
        elif rank == M - 1: fmm_count += 1
        else: rem_count += 1
            
    chisq = (((fm_count - p_full_rank * num_blocks)**2) / (p_full_rank * num_blocks) +
             ((fmm_count - p_rank_m1 * num_blocks)**2) / (p_rank_m1 * num_blocks) +
             ((rem_count - p_remainder * num_blocks)**2) / (p_remainder * num_blocks))
    
    p_value = math.exp(-chisq / 2.0)
    success = (p_value >= 0.01)
    return [chisq, p_value, success]

# --- Cell 6: Spectral (DFT) Test ---
def spectral_test(input_str, n):
    """
    Performs the NIST Discrete Fourier Transform (DFT) / Spectral Test.
    """
    ts = [(1 if bit == '1' else -1) for bit in input_str]
    ts_np = np.array(ts)
    fs = np.fft.fft(ts_np)
    mags = abs(fs)[:n//2]
    T = math.sqrt(math.log(1.0 / 0.05) * n)
    N0 = 0.95 * (n / 2.0)
    N1 = float(np.sum(mags < T))
    d = (N1 - N0) / math.sqrt((n * 0.95 * 0.05) / 4.0)
    p_value = math.erfc(abs(d) / math.sqrt(2))
    success = (p_value >= 0.01)
    return [N0, N1, d, p_value, success]

# --- Cell 7: Non-Overlapping Template Test ---
def non_overlapping_template_test(input_str):
    """
    Performs the NIST Non-Overlapping Template Matching Test.
    """
    n = len(input_str)
    templates_int = [ [0, 1], [1, 0], [0, 0, 1], [0, 1, 1], [1, 0, 0], [1, 1, 0],
                      [0, 0, 0, 1], [0, 0, 1, 1], [0, 1, 1, 1], [1, 0, 0, 0], [1, 1, 0, 0], [1, 1, 1, 0] ]
    B_int = templates_int[0] # Using a fixed template '01' for consistency
    m = len(B_int)
    template_str = "".join(map(str, B_int))
    N = 8    # The test is run on N blocks
    M = n // N # Length of each block
    
    if M < 21:
        return [0.0, 0.0, 0.0, 0.0, False]

    W = [] 
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        count = 0
        position = 0
        while position < (M - m + 1):
            if block[position : position + m] == template_str:
                count += 1
                position += m
            else:
                position += 1
        W.append(count)

    mu = (M - m + 1) / (2**m)
    sigma_sq = M * ((1.0 / (2**m)) - ((2.0 * m - 1.0) / (2**(2 * m))))
    chi_sq = 0.0
    for count in W:
        chi_sq += ((count - mu)**2) / sigma_sq
        
    p_value = ss.gammaincc(N / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [mu, sigma_sq, chi_sq, p_value, success, template_str]

# --- Cell 8: Overlapping Template Test (and helpers) ---
def lgamma(x):
    return math.log(ss.gamma(x))

def Pr(u, eta):
    if u == 0:
        return math.exp(-eta)
    else:
        sum_val = 0.0
        for l in range(1, u + 1):
            sum_val += math.exp(-eta - u * math.log(2) + l * math.log(eta) - lgamma(l + 1) + lgamma(u) - lgamma(l) - lgamma(u - l + 1))
        return sum_val

def overlapping_template_test(input_str):
    """
    Performs the NIST Overlapping Template Matching Test.
    """
    n = len(input_str)
    m = 10   # Length of the template pattern
    N = 968  # Number of blocks
    M = 1032 # Length of each block
    K = 5    # Degrees of freedom
    template_str = '1' * m
    
    if n < (M * N):
        return [0.0, 0.0, False, [0]*6]

    v = [0] * (K + 1)
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        count = 0
        for j in range(M - m + 1):
            if block[j : j + m] == template_str:
                count += 1
        
        if count >= K:
            v[K] += 1
        else:
            v[count] += 1

    lambd = (M - m + 1.0) / (2.0**m)
    eta = lambd / 2.0
    pi = [Pr(i, eta) for i in range(K)]
    pi.append(1.0 - sum(pi)) 

    chisq = 0.0
    for i in range(K + 1):
        chisq += ((v[i] - N * pi[i])**2) / (N * pi[i])
        
    p_value = ss.gammaincc((K / 2.0), chisq / 2.0)
    success = (p_value >= 0.01)
    return [chisq, p_value, success, v]

# --- Cell 9: Universal Test ---
def universal_test(input_str):
    """
    Performs Maurer's Universal Statistical Test.
    """
    n = len(input_str)
    if n >= 1059061760: L = 16
    elif n >= 496435200:  L = 15
    elif n >= 231669760:  L = 14
    elif n >= 107560960:  L = 13
    elif n >= 496435200:  L = 12
    elif n >= 22753280:   L = 11
    elif n >= 10342400:   L = 10
    elif n >= 4654080:    L = 9
    elif n >= 2068480:    L = 8
    elif n >= 904960:     L = 7
    elif n >= 387840:     L = 6
    else:
        return [0] * 3 

    num_blocks = math.floor(n / L)
    Q = 10 * (2**L)
    K = num_blocks - Q
    
    if K <= 0:
        return [0] * 3

    num_symbols = 2**L
    T = [0] * num_symbols
    for i in range(Q):
        pattern = input_str[i * L : (i + 1) * L]
        idx = int(pattern, 2)
        T[idx] = i + 1

    log_sum = 0.0
    for i in range(Q, num_blocks):
        pattern = input_str[i * L : (i + 1) * L]
        j = int(pattern, 2)
        distance = i + 1 - T[j]
        T[j] = i + 1
        log_sum += math.log2(distance)

    fn = log_sum / K
    ev_table = [0, 0.73264948, 1.5374383, 2.40160681, 3.31122472,
                4.25342659, 5.2177052, 6.1962507, 7.1836656,
                8.1764248, 9.1723243, 10.170032, 11.168765,
                12.168070, 13.167693, 14.167488, 15.167379]
    var_table = [0, 0.690, 1.338, 1.901, 2.358, 2.705, 2.954, 3.125,
                 3.238, 3.311, 3.356, 3.384, 3.401, 3.410, 3.416,
                 3.419, 3.421]
                 
    expected_value = ev_table[L]
    variance = var_table[L]
    
    arg = abs(fn - expected_value) / (math.sqrt(2 * variance))
    p_value = math.erfc(arg)
    success = (p_value >= 0.01)
    return [fn, p_value, success]

# --- Cell 10: Linear Complexity Test (and helpers) ---
def padding(input_str, n):
    while len(input_str) < n:
        input_str = '0' + input_str
    return input_str

def berlekamp_massey(input_str):
    n = len(input_str)
    b = '1' + '0' * (n - 1)
    c = '1' + '0' * (n - 1)
    L = 0
    m = -1
    N = 0
    while N < n:
        d = int(input_str[N], 2)
        if L > 0:
            k_str = c[1 : L + 1]
            h_str = input_str[N - L : N][::-1]
            k_int = int(k_str, 2)
            h_int = int(h_str, 2)
            r = bin(k_int & h_int)[2:].count('1')
            d = d ^ (r % 2)

        if d != 0:
            t = c
            k_str = c[N - m : n]
            k_int = int(k_str, 2)
            h_str = b[0 : n - N + m]
            h_int = int(h_str, 2)
            k_int = k_int ^ h_int
            c = c[0 : N - m] + padding(bin(k_int)[2:], n - N + m)
            if L <= (N / 2):
                L = N + 1 - L
                m = N
                b = t
        N += 1
    return L, c[0:L]

def linear_complexity_test(input_str, M=512):
    n = len(input_str)
    K = 6  # Degrees of freedom
    N = math.floor(n / M)
    if n < 1000000:
        return [0.0, 0.0, False, [0]*7]

    LC = [berlekamp_massey(input_str[i * M : (i + 1) * M])[0] for i in range(N)]
    mu = (M / 2.0) + ((((-1)**(M + 1)) + 9.0) / 36.0) - (((M / 3.0) + (2.0 / 9.0)) / (2**M))
    T = [(((-1.0)**M) * (lc - mu) + (2.0 / 9.0)) for lc in LC]
    
    v = [0] * (K + 1)
    for t in T:
        if   t <= -2.5: v[0] += 1
        elif t <= -1.5: v[1] += 1
        elif t <= -0.5: v[2] += 1
        elif t <= 0.5:  v[3] += 1
        elif t <= 1.5:  v[4] += 1
        elif t <= 2.5:  v[5] += 1
        else:           v[6] += 1

    pi = [0.010417, 0.03125, 0.125, 0.5, 0.25, 0.0625, 0.020833]
    chi_sq = sum(((v[i] - N * pi[i])**2.0) / (N * pi[i]) for i in range(K + 1))
    p_value = ss.gammaincc(K / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [chi_sq, p_value, success, v]

# --- Cell 11: Serial Test (and helpers) ---
def int_to_pattern_str(n, m):
    return bin(n)[2:].zfill(m)

def psi_sq(m, n, padded_input):
    if m == 0:
        return 0.0
    counts = [0] * (2**m)
    for i in range(n):
        pattern_str = padded_input[i : i + m]
        idx = int(pattern_str, 2)
        counts[idx] += 1
    
    psi_sq_m = 0.0
    for count in counts:
        psi_sq_m += count**2
        
    psi_sq_m = (psi_sq_m * (2**m) / n) - n
    return psi_sq_m

def serial_test(input_str, patternlen=None):
    n = len(input_str)
    if patternlen is not None:
        m = patternlen
    else:
        m = math.floor(math.log2(n)) - 2
        if m < 1: m = 1
    
    padded_input = input_str + input_str[0 : m - 1]
    
    psi_sq_m = psi_sq(m, n, padded_input)
    psi_sq_m_minus_1 = psi_sq(m - 1, n, padded_input)
    psi_sq_m_minus_2 = psi_sq(m - 2, n, padded_input)
    
    delta1 = psi_sq_m - psi_sq_m_minus_1
    delta2 = psi_sq_m - (2 * psi_sq_m_minus_1) + psi_sq_m_minus_2

    p_value1 = ss.gammaincc(2**(m - 2), delta1 / 2.0)
    p_value2 = ss.gammaincc(2**(m - 3), delta2 / 2.0)
    
    success = (p_value1 >= 0.01) and (p_value2 >= 0.01)
    return [delta1, delta2, p_value1, p_value2, success]

# --- Cell 12: Approximate Entropy Test ---
def approximate_entropy_test(input_str):
    n = len(input_str)
    m = math.floor(math.log2(n)) - 6
    if m < 2: m = 2

    phi = [0.0, 0.0]
    for j in range(m, m + 2):
        if j == 0:
            phi[j-m] = 0.0
            continue
            
        padded_input = input_str + input_str[0 : j - 1]
        counts = [0] * (2**j)
        for i in range(n):
            pattern = padded_input[i : i + j]
            idx = int(pattern, 2)
            counts[idx] += 1
            
        probabilities = [count / n for count in counts]
        
        temp_phi = 0.0
        for prob in probabilities:
            if prob > 0:
                temp_phi += prob * math.log(prob)
        phi[j-m] = temp_phi

    appen_m = phi[0] - phi[1]
    chi_sq = 2 * n * (math.log(2) - appen_m)
    p_value = ss.gammaincc(2**(m - 1), chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [appen_m, chi_sq, p_value, success]

# --- Cell 13: Cumulative Sums Test (and helpers) ---
def normcdf(n):
    return 0.5 * math.erfc(-n * math.sqrt(0.5))

def p_value(n, z):
    sum_a = 0.0
    start_k = int(math.floor((((float(-n) / z) + 1.0) / 4.0)))
    end_k = int(math.floor((((float(n) / z) - 1.0) / 4.0)))
    for k in range(start_k, end_k + 1):
        c1 = (((4.0 * k) + 1.0) * z) / math.sqrt(n)
        d1 = normcdf(c1)
        c2 = (((4.0 * k) - 1.0) * z) / math.sqrt(n)
        e1 = normcdf(c2)
        sum_a = sum_a + d1 - e1

    sum_b = 0.0
    start_k = int(math.floor((((float(-n) / z) - 3.0) / 4.0)))
    end_k = int(math.floor((((float(n) / z) - 1.0) / 4.0)))
    for k in range(start_k, end_k + 1):
        c1 = (((4.0 * k) + 3.0) * z) / math.sqrt(n)
        d1 = normcdf(c1)
        c2 = (((4.0 * k) + 1.0) * z) / math.sqrt(n)
        e1 = normcdf(c2)
        sum_b = sum_b + d1 - e1

    p = 1.0 - sum_a + sum_b
    return p

def cumulative_sums_test(input_str):
    n = len(input_str)
    x = [(int(bit) * 2 - 1) for bit in input_str]
    
    pos = 0; forward_max = 0
    for e in x:
        pos += e
        if abs(pos) > forward_max:
            forward_max = abs(pos)
    p_forward = p_value(n, forward_max)
    
    pos = 0; backward_max = 0
    for e in reversed(x):
        pos += e
        if abs(pos) > backward_max:
            backward_max = abs(pos)
    p_backward = p_value(n, backward_max)
    
    success = (p_forward >= 0.01) and (p_backward >= 0.01)
    return [p_forward, p_backward, success]

# --- Cell 14: Random Excursions Test ---
def random_excursions_test(input_str):
    n = len(input_str)
    if n < 1000000:
        return [False, None] # Not enough data

    x_np = np.array(list(input_str), dtype=np.int8) * 2 - 1
    s = np.cumsum(x_np)
    s_prime = np.concatenate(([0], s, [0]))
    zero_crossings = np.where(s_prime == 0)[0]
    cycles = [s_prime[zero_crossings[i]:zero_crossings[i+1]+1] for i in range(len(zero_crossings) - 1)]
    J = len(cycles)
    
    if J < 10:
        return [False, None] # Not enough cycles

    states = [-4, -3, -2, -1, 1, 2, 3, 4]
    pi_kx = [
        [0.5, 0.25, 0.125, 0.0625, 0.03125, 0.03125],
        [0.75, 0.0625, 0.046875, 0.03515625, 0.0263671875, 0.0791015625],
        [0.8333333333, 0.0277777778, 0.0231481481, 0.0192901235, 0.0160751029, 0.0803755123],
        [0.875, 0.015625, 0.013671875, 0.0119628906, 0.0104675293, 0.0732727051],
    ]

    results = []
    overall_success = True
    
    v_counts = {state: [0] * 6 for state in states}
    for cycle in cycles:
        cycle_counter = Counter(cycle)
        for x_state in states:
            count = cycle_counter.get(x_state, 0)
            if count >= 5: v_counts[x_state][5] += 1
            else: v_counts[x_state][count] += 1
    
    for x_state in states:
        v_xk = v_counts[x_state]
        pi = pi_kx[abs(x_state) - 1]
        chi_sq = 0.0
        for k in range(6):
            numerator = (v_xk[k] - J * pi[k])**2
            denominator = J * pi[k]
            if denominator == 0: continue
            chi_sq += numerator / denominator
            
        p_value = ss.gammaincc(5.0 / 2.0, chi_sq / 2.0)
        if p_value < 0.01: overall_success = False
        results.append({'state': x_state, 'p_value': p_value, 'pass': (p_value >= 0.01)})

    return overall_success, results

# --- Cell 16: Random Excursions Variant Test ---
def random_excursions_variant_test(input_str):
    n = len(input_str)
    if n < 1000000:
        return [False, 0, None] # Not enough data

    x_np = np.array(list(input_str), dtype=np.int8) * 2 - 1
    s = np.cumsum(x_np)
    s_prime = np.concatenate(([0], s))
    J = np.count_nonzero(s_prime == 0)

    if J < 10:
        return [False, J, None] # Not enough cycles

    states = list(range(-9, 0)) + list(range(1, 10))
    results = []
    overall_success = True
    
    state_counts = Counter(s_prime)
    
    for x_state in states:
        count_x = state_counts.get(x_state, 0)
        numerator = abs(count_x - J)
        denominator = math.sqrt(2.0 * J * (4.0 * abs(x_state) - 2.0))
        
        if denominator == 0:
            p_value = 0.0
        else:
            p_value = ss.erfc(numerator / denominator)
        
        if p_value < 0.01:
            overall_success = False
        
        results.append({'state': x_state, 'count': count_x, 'p_value': p_value, 'pass': (p_value >= 0.01)})
        
    return overall_success, J, results

# ===================================================
# MAIN EXECUTION BLOCK — DIRECT .BIN TESTING
# ===================================================
if __name__ == "__main__":

    BIN_FILE = "cnn_csprng_output.bin"
    BITS_TO_TEST = 1_280_000   # >= 1e6 required for full NIST

    print(f"Reading binary data from '{BIN_FILE}'...")

    try:
        with open(BIN_FILE, "rb") as f:
            byte_data = f.read()

        byte_array = np.frombuffer(byte_data, dtype=np.uint8)
        bit_array = np.unpackbits(byte_array)

        if len(bit_array) < BITS_TO_TEST:
            raise ValueError(
                f"Not enough bits: need {BITS_TO_TEST}, got {len(bit_array)}"
            )

        bit_array = bit_array[:BITS_TO_TEST]
        binary_data = "".join(bit_array.astype(str))
        data_length = len(binary_data)

        print(f"Loaded {data_length} bits successfully.")

    except Exception as e:
        print("ERROR reading .bin file:", e)
        sys.exit(1)

    print("\nRunning all 15 NIST Statistical Tests...")
    print("=" * 70)
    print(f"{'Test Name':<30} | {'P-Value(s)':<25} | {'Result'}")
    print("-" * 70)

    all_tests_passed = True

    def report(name, passed, pvals):
        print(f"{name:<30} | {pvals:<25} | {'PASS' if passed else 'FAIL'}")
        return passed

    # 1. Frequency
    r = frequency_test(binary_data, data_length)
    report("Frequency", r[4], f"{r[3]:.6f}")

    # 2. Block Frequency
    r = block_frequency_test(binary_data, data_length)
    report("Block Frequency", r[2], f"{r[1]:.6f}")

    # 3. Runs
    r = runs_test(binary_data, data_length)
    report("Runs", r[5], f"{r[4]:.6f}")

    # 4. Longest Run of Ones
    r = longest_run_test(binary_data)
    report("Longest Run", r[2], f"{r[1]:.6f}")

    # 5. Rank
    r = binary_matrix_rank_test(binary_data, data_length)
    report("Binary Matrix Rank", r[2], f"{r[1]:.6f}")

    # 6. Spectral (DFT)
    r = spectral_test(binary_data, data_length)
    report("Spectral (DFT)", r[4], f"{r[3]:.6f}")

    # 7. Non-overlapping Template
    r = non_overlapping_template_test(binary_data)
    report("Non-overlap Template", r[4], f"{r[3]:.6f}")

    # 8. Overlapping Template
    r = overlapping_template_test(binary_data)
    report("Overlapping Template", r[2], f"{r[1]:.6f}")

    # 9. Universal
    r = universal_test(binary_data)
    report("Universal", r[2], f"{r[1]:.6f}")

    # 10. Linear Complexity
    r = linear_complexity_test(binary_data)
    report("Linear Complexity", r[2], f"{r[1]:.6f}")

    # 11. Serial
    r = serial_test(binary_data, 10)
    report("Serial", r[4], f"{r[2]:.6f}, {r[3]:.6f}")

    # 12. Approximate Entropy
    r = approximate_entropy_test(binary_data)
    report("Approximate Entropy", r[3], f"{r[2]:.6f}")

    # 13. Cumulative Sums
    r = cumulative_sums_test(binary_data)
    report("Cumulative Sums", r[2], f"{r[0]:.6f}, {r[1]:.6f}")

    # 14. Random Excursions
    passed, res = random_excursions_test(binary_data)
    if res is None:
        report("Random Excursions", False, "Insufficient cycles")
    else:
        min_p = min(x["p_value"] for x in res)
        report("Random Excursions", passed, f"{min_p:.6f}")

    # 15. Random Excursions Variant
    passed, J, res = random_excursions_variant_test(binary_data)
    if res is None:
        report("Random Excursions Var", False, "Insufficient cycles")
    else:
        min_p = min(x["p_value"] for x in res)
        report("Random Excursions Var", passed, f"{min_p:.6f}")

    print("=" * 70)


Successfully imported gf2matrix.py
Reading binary data from 'cnn_csprng_output.bin'...
Loaded 1280000 bits successfully.

Running all 15 NIST Statistical Tests...
Test Name                      | P-Value(s)                | Result
----------------------------------------------------------------------
Frequency                      | 0.826492                  | PASS
Block Frequency                | 0.717329                  | PASS
Runs                           | 0.877737                  | PASS
Longest Run                    | 0.899119                  | PASS
Binary Matrix Rank             | 0.079493                  | PASS
Spectral (DFT)                 | 0.450651                  | PASS
Non-overlap Template           | 0.845422                  | PASS
Overlapping Template           | 0.212552                  | PASS
Universal                      | 0.999548                  | PASS
Linear Complexity              | 0.601695                  | PASS
Serial                         | 0.087

In [3]:
import math
import scipy.special as ss
from fractions import Fraction
import os
import numpy as np
from collections import Counter
import sys

try:
    import gf2matrix
    print("Successfully imported gf2matrix.py")
except ImportError:
    print("ERROR: Could not import 'gf2matrix'.")
    print("Please make sure the 'gf2matrix.py' file is in the same directory as this script.")
    sys.exit()

# --- Cell 1: Frequency (Monobit) Test ---
def frequency_test(input_str, n):
    """
    Performs a frequency test on a binary string.
    """
    ones = input_str.count('1')
    zeroes = input_str.count('0')
    s = abs(ones - zeroes)
    p = math.erfc(float(s)/(math.sqrt(float(n)) * math.sqrt(2.0)))
    success = (p >= 0.01)
    return [zeroes, ones, s, p, success]

# --- Cell 2: Block Frequency Test ---
def block_frequency_test(input_str, n, M=128):
    """
    Performs the Block Frequency Test on a binary string.
    """
    num_blocks = math.floor(n / M)
    block_size = M
    
    if n < 100 or num_blocks < 1:
        return [0.0, 0.0, False]

    proportions = []
    for i in range(num_blocks):
        block = input_str[i * block_size : (i + 1) * block_size]
        ones = block.count('1')
        proportions.append(Fraction(ones, block_size))

    chisq = 0.0
    for prop in proportions:
        chisq += 4.0 * block_size * ((prop - Fraction(1, 2))**2)
    
    p_value = ss.gammaincc(num_blocks / 2.0, float(chisq) / 2.0)
    success = (p_value >= 0.01)
    return [float(chisq), p_value, success]

# --- Cell 3: Runs Test ---
def runs_test(input_str, n):
    """
    Performs the NIST Runs Test on a binary string.
    """
    ones = input_str.count('1')
    zeroes = input_str.count('0')
    
    prop = float(ones) / float(n)
    tau = 2.0 / math.sqrt(n)
    if abs(prop - 0.5) >= tau:
        return [zeroes, ones, prop, 0.0, 0.0, False]

    vobs = 1.0
    for i in range(n - 1):
        if input_str[i] != input_str[i+1]:
            vobs += 1.0

    p_value = math.erfc(abs(vobs - (2.0 * n * prop * (1.0 - prop))) / 
                       (2.0 * math.sqrt(2.0 * n) * prop * (1.0 - prop)))
    success = (p_value >= 0.01)
    return [zeroes, ones, prop, vobs, p_value, success]

# --- Cell 4: Longest Run of Ones Test ---
def longest_run_test(input_str):
    """
    Performs the NIST Longest Run of Ones in a Block Test. (M=8)
    """
    n = len(input_str)
    M = 8   # Block size
    K = 3   # Degrees of freedom
    PI = [0.2148, 0.3672, 0.2305, 0.1875] 
    N = math.floor(n / M)
    
    if N < 16:
        return [0.0, 0.0, False]

    v = [0, 0, 0, 0]
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        current_run = 0
        longest_run = 0
        for bit in block:
            if bit == '1':
                current_run += 1
                if current_run > longest_run:
                    longest_run = current_run
            else:
                current_run = 0
        
        if longest_run <= 1: v[0] += 1
        elif longest_run == 2: v[1] += 1
        elif longest_run == 3: v[2] += 1
        else: v[3] += 1
    
    chi_sq = 0.0
    for i in range(K + 1):
        numerator = (v[i] - N * PI[i])**2
        denominator = N * PI[i]
        chi_sq += numerator / denominator
        
    p_value = ss.gammaincc(K / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [chi_sq, p_value, success]

# --- Cell 5: Binary Matrix Rank Test ---
def binary_matrix_rank_test(input_str, n, M=32, Q=32):
    """
    Performs the NIST Binary Matrix Rank Test. Requires gf2matrix.py
    """
    num_blocks = int(math.floor(n / (M * Q)))
    if num_blocks < 38:
        return [0.0, 0.0, False]

    product = 1.0
    for i in range(M):
        product *= (1.0 - 2.0**(i - Q)) * (1.0 - 2.0**(i - M)) / (1.0 - 2.0**(i - M))
    p_full_rank = product * (2.0**(M * (Q + M - M) - (M * Q)))

    product = 1.0
    for i in range(M - 1):
        product *= (1.0 - 2.0**(i - Q)) * (1.0 - 2.0**(i - M)) / (1.0 - 2.0**(i - (M-1)))
    p_rank_m1 = product * (2.0**((M-1)*(Q+M-(M-1)) - M*Q))
    
    p_remainder = 1.0 - p_full_rank - p_rank_m1
    fm_count = 0; fmm_count = 0; rem_count = 0

    for blk_num in range(num_blocks):
        block_str = input_str[blk_num*M*Q : (blk_num+1)*M*Q]
        block = [int(bit) for bit in block_str]
        
        matrix = gf2matrix.matrix_from_bits(M, Q, block, blk_num)
        rank = gf2matrix.rank(M, Q, matrix, blk_num)

        if rank == M: fm_count += 1
        elif rank == M - 1: fmm_count += 1
        else: rem_count += 1
            
    chisq = (((fm_count - p_full_rank * num_blocks)**2) / (p_full_rank * num_blocks) +
             ((fmm_count - p_rank_m1 * num_blocks)**2) / (p_rank_m1 * num_blocks) +
             ((rem_count - p_remainder * num_blocks)**2) / (p_remainder * num_blocks))
    
    p_value = math.exp(-chisq / 2.0)
    success = (p_value >= 0.01)
    return [chisq, p_value, success]

# --- Cell 6: Spectral (DFT) Test ---
def spectral_test(input_str, n):
    """
    Performs the NIST Discrete Fourier Transform (DFT) / Spectral Test.
    """
    ts = [(1 if bit == '1' else -1) for bit in input_str]
    ts_np = np.array(ts)
    fs = np.fft.fft(ts_np)
    mags = abs(fs)[:n//2]
    T = math.sqrt(math.log(1.0 / 0.05) * n)
    N0 = 0.95 * (n / 2.0)
    N1 = float(np.sum(mags < T))
    d = (N1 - N0) / math.sqrt((n * 0.95 * 0.05) / 4.0)
    p_value = math.erfc(abs(d) / math.sqrt(2))
    success = (p_value >= 0.01)
    return [N0, N1, d, p_value, success]

# --- Cell 7: Non-Overlapping Template Test ---
def non_overlapping_template_test(input_str):
    """
    Performs the NIST Non-Overlapping Template Matching Test.
    """
    n = len(input_str)
    templates_int = [ [0, 1], [1, 0], [0, 0, 1], [0, 1, 1], [1, 0, 0], [1, 1, 0],
                      [0, 0, 0, 1], [0, 0, 1, 1], [0, 1, 1, 1], [1, 0, 0, 0], [1, 1, 0, 0], [1, 1, 1, 0] ]
    B_int = templates_int[0] # Using a fixed template '01' for consistency
    m = len(B_int)
    template_str = "".join(map(str, B_int))
    N = 8    # The test is run on N blocks
    M = n // N # Length of each block
    
    if M < 21:
        return [0.0, 0.0, 0.0, 0.0, False]

    W = [] 
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        count = 0
        position = 0
        while position < (M - m + 1):
            if block[position : position + m] == template_str:
                count += 1
                position += m
            else:
                position += 1
        W.append(count)

    mu = (M - m + 1) / (2**m)
    sigma_sq = M * ((1.0 / (2**m)) - ((2.0 * m - 1.0) / (2**(2 * m))))
    chi_sq = 0.0
    for count in W:
        chi_sq += ((count - mu)**2) / sigma_sq
        
    p_value = ss.gammaincc(N / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [mu, sigma_sq, chi_sq, p_value, success, template_str]

# --- Cell 8: Overlapping Template Test (and helpers) ---
def lgamma(x):
    return math.log(ss.gamma(x))

def Pr(u, eta):
    if u == 0:
        return math.exp(-eta)
    else:
        sum_val = 0.0
        for l in range(1, u + 1):
            sum_val += math.exp(-eta - u * math.log(2) + l * math.log(eta) - lgamma(l + 1) + lgamma(u) - lgamma(l) - lgamma(u - l + 1))
        return sum_val

def overlapping_template_test(input_str):
    """
    Performs the NIST Overlapping Template Matching Test.
    """
    n = len(input_str)
    m = 10   # Length of the template pattern
    N = 968  # Number of blocks
    M = 1032 # Length of each block
    K = 5    # Degrees of freedom
    template_str = '1' * m
    
    if n < (M * N):
        return [0.0, 0.0, False, [0]*6]

    v = [0] * (K + 1)
    for i in range(N):
        block = input_str[i * M : (i + 1) * M]
        count = 0
        for j in range(M - m + 1):
            if block[j : j + m] == template_str:
                count += 1
        
        if count >= K:
            v[K] += 1
        else:
            v[count] += 1

    lambd = (M - m + 1.0) / (2.0**m)
    eta = lambd / 2.0
    pi = [Pr(i, eta) for i in range(K)]
    pi.append(1.0 - sum(pi)) 

    chisq = 0.0
    for i in range(K + 1):
        chisq += ((v[i] - N * pi[i])**2) / (N * pi[i])
        
    p_value = ss.gammaincc((K / 2.0), chisq / 2.0)
    success = (p_value >= 0.01)
    return [chisq, p_value, success, v]

# --- Cell 9: Universal Test ---
def universal_test(input_str):
    """
    Performs Maurer's Universal Statistical Test.
    """
    n = len(input_str)
    if n >= 1059061760: L = 16
    elif n >= 496435200:  L = 15
    elif n >= 231669760:  L = 14
    elif n >= 107560960:  L = 13
    elif n >= 496435200:  L = 12
    elif n >= 22753280:   L = 11
    elif n >= 10342400:   L = 10
    elif n >= 4654080:    L = 9
    elif n >= 2068480:    L = 8
    elif n >= 904960:     L = 7
    elif n >= 387840:     L = 6
    else:
        return [0] * 3 

    num_blocks = math.floor(n / L)
    Q = 10 * (2**L)
    K = num_blocks - Q
    
    if K <= 0:
        return [0] * 3

    num_symbols = 2**L
    T = [0] * num_symbols
    for i in range(Q):
        pattern = input_str[i * L : (i + 1) * L]
        idx = int(pattern, 2)
        T[idx] = i + 1

    log_sum = 0.0
    for i in range(Q, num_blocks):
        pattern = input_str[i * L : (i + 1) * L]
        j = int(pattern, 2)
        distance = i + 1 - T[j]
        T[j] = i + 1
        log_sum += math.log2(distance)

    fn = log_sum / K
    ev_table = [0, 0.73264948, 1.5374383, 2.40160681, 3.31122472,
                4.25342659, 5.2177052, 6.1962507, 7.1836656,
                8.1764248, 9.1723243, 10.170032, 11.168765,
                12.168070, 13.167693, 14.167488, 15.167379]
    var_table = [0, 0.690, 1.338, 1.901, 2.358, 2.705, 2.954, 3.125,
                 3.238, 3.311, 3.356, 3.384, 3.401, 3.410, 3.416,
                 3.419, 3.421]
                 
    expected_value = ev_table[L]
    variance = var_table[L]
    
    arg = abs(fn - expected_value) / (math.sqrt(2 * variance))
    p_value = math.erfc(arg)
    success = (p_value >= 0.01)
    return [fn, p_value, success]

# --- Cell 10: Linear Complexity Test (and helpers) ---
def padding(input_str, n):
    while len(input_str) < n:
        input_str = '0' + input_str
    return input_str

def berlekamp_massey(input_str):
    n = len(input_str)
    b = '1' + '0' * (n - 1)
    c = '1' + '0' * (n - 1)
    L = 0
    m = -1
    N = 0
    while N < n:
        d = int(input_str[N], 2)
        if L > 0:
            k_str = c[1 : L + 1]
            h_str = input_str[N - L : N][::-1]
            k_int = int(k_str, 2)
            h_int = int(h_str, 2)
            r = bin(k_int & h_int)[2:].count('1')
            d = d ^ (r % 2)

        if d != 0:
            t = c
            k_str = c[N - m : n]
            k_int = int(k_str, 2)
            h_str = b[0 : n - N + m]
            h_int = int(h_str, 2)
            k_int = k_int ^ h_int
            c = c[0 : N - m] + padding(bin(k_int)[2:], n - N + m)
            if L <= (N / 2):
                L = N + 1 - L
                m = N
                b = t
        N += 1
    return L, c[0:L]

def linear_complexity_test(input_str, M=512):
    n = len(input_str)
    K = 6  # Degrees of freedom
    N = math.floor(n / M)
    if n < 1000000:
        return [0.0, 0.0, False, [0]*7]

    LC = [berlekamp_massey(input_str[i * M : (i + 1) * M])[0] for i in range(N)]
    mu = (M / 2.0) + ((((-1)**(M + 1)) + 9.0) / 36.0) - (((M / 3.0) + (2.0 / 9.0)) / (2**M))
    T = [(((-1.0)**M) * (lc - mu) + (2.0 / 9.0)) for lc in LC]
    
    v = [0] * (K + 1)
    for t in T:
        if   t <= -2.5: v[0] += 1
        elif t <= -1.5: v[1] += 1
        elif t <= -0.5: v[2] += 1
        elif t <= 0.5:  v[3] += 1
        elif t <= 1.5:  v[4] += 1
        elif t <= 2.5:  v[5] += 1
        else:           v[6] += 1

    pi = [0.010417, 0.03125, 0.125, 0.5, 0.25, 0.0625, 0.020833]
    chi_sq = sum(((v[i] - N * pi[i])**2.0) / (N * pi[i]) for i in range(K + 1))
    p_value = ss.gammaincc(K / 2.0, chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [chi_sq, p_value, success, v]

# --- Cell 11: Serial Test (and helpers) ---
def int_to_pattern_str(n, m):
    return bin(n)[2:].zfill(m)

def psi_sq(m, n, padded_input):
    if m == 0:
        return 0.0
    counts = [0] * (2**m)
    for i in range(n):
        pattern_str = padded_input[i : i + m]
        idx = int(pattern_str, 2)
        counts[idx] += 1
    
    psi_sq_m = 0.0
    for count in counts:
        psi_sq_m += count**2
        
    psi_sq_m = (psi_sq_m * (2**m) / n) - n
    return psi_sq_m

def serial_test(input_str, patternlen=None):
    n = len(input_str)
    if patternlen is not None:
        m = patternlen
    else:
        m = math.floor(math.log2(n)) - 2
        if m < 1: m = 1
    
    padded_input = input_str + input_str[0 : m - 1]
    
    psi_sq_m = psi_sq(m, n, padded_input)
    psi_sq_m_minus_1 = psi_sq(m - 1, n, padded_input)
    psi_sq_m_minus_2 = psi_sq(m - 2, n, padded_input)
    
    delta1 = psi_sq_m - psi_sq_m_minus_1
    delta2 = psi_sq_m - (2 * psi_sq_m_minus_1) + psi_sq_m_minus_2

    p_value1 = ss.gammaincc(2**(m - 2), delta1 / 2.0)
    p_value2 = ss.gammaincc(2**(m - 3), delta2 / 2.0)
    
    success = (p_value1 >= 0.01) and (p_value2 >= 0.01)
    return [delta1, delta2, p_value1, p_value2, success]

# --- Cell 12: Approximate Entropy Test ---
def approximate_entropy_test(input_str):
    n = len(input_str)
    m = math.floor(math.log2(n)) - 6
    if m < 2: m = 2

    phi = [0.0, 0.0]
    for j in range(m, m + 2):
        if j == 0:
            phi[j-m] = 0.0
            continue
            
        padded_input = input_str + input_str[0 : j - 1]
        counts = [0] * (2**j)
        for i in range(n):
            pattern = padded_input[i : i + j]
            idx = int(pattern, 2)
            counts[idx] += 1
            
        probabilities = [count / n for count in counts]
        
        temp_phi = 0.0
        for prob in probabilities:
            if prob > 0:
                temp_phi += prob * math.log(prob)
        phi[j-m] = temp_phi

    appen_m = phi[0] - phi[1]
    chi_sq = 2 * n * (math.log(2) - appen_m)
    p_value = ss.gammaincc(2**(m - 1), chi_sq / 2.0)
    success = (p_value >= 0.01)
    return [appen_m, chi_sq, p_value, success]

# --- Cell 13: Cumulative Sums Test (and helpers) ---
def normcdf(n):
    return 0.5 * math.erfc(-n * math.sqrt(0.5))

def p_value(n, z):
    sum_a = 0.0
    start_k = int(math.floor((((float(-n) / z) + 1.0) / 4.0)))
    end_k = int(math.floor((((float(n) / z) - 1.0) / 4.0)))
    for k in range(start_k, end_k + 1):
        c1 = (((4.0 * k) + 1.0) * z) / math.sqrt(n)
        d1 = normcdf(c1)
        c2 = (((4.0 * k) - 1.0) * z) / math.sqrt(n)
        e1 = normcdf(c2)
        sum_a = sum_a + d1 - e1

    sum_b = 0.0
    start_k = int(math.floor((((float(-n) / z) - 3.0) / 4.0)))
    end_k = int(math.floor((((float(n) / z) - 1.0) / 4.0)))
    for k in range(start_k, end_k + 1):
        c1 = (((4.0 * k) + 3.0) * z) / math.sqrt(n)
        d1 = normcdf(c1)
        c2 = (((4.0 * k) + 1.0) * z) / math.sqrt(n)
        e1 = normcdf(c2)
        sum_b = sum_b + d1 - e1

    p = 1.0 - sum_a + sum_b
    return p

def cumulative_sums_test(input_str):
    n = len(input_str)
    x = [(int(bit) * 2 - 1) for bit in input_str]
    
    pos = 0; forward_max = 0
    for e in x:
        pos += e
        if abs(pos) > forward_max:
            forward_max = abs(pos)
    p_forward = p_value(n, forward_max)
    
    pos = 0; backward_max = 0
    for e in reversed(x):
        pos += e
        if abs(pos) > backward_max:
            backward_max = abs(pos)
    p_backward = p_value(n, backward_max)
    
    success = (p_forward >= 0.01) and (p_backward >= 0.01)
    return [p_forward, p_backward, success]

# --- Cell 14: Random Excursions Test ---
def random_excursions_test(input_str):
    n = len(input_str)
    if n < 1000000:
        return [False, None] # Not enough data

    x_np = np.array(list(input_str), dtype=np.int8) * 2 - 1
    s = np.cumsum(x_np)
    s_prime = np.concatenate(([0], s, [0]))
    zero_crossings = np.where(s_prime == 0)[0]
    cycles = [s_prime[zero_crossings[i]:zero_crossings[i+1]+1] for i in range(len(zero_crossings) - 1)]
    J = len(cycles)
    
    if J < 10:
        return [False, None] # Not enough cycles

    states = [-4, -3, -2, -1, 1, 2, 3, 4]
    pi_kx = [
        [0.5, 0.25, 0.125, 0.0625, 0.03125, 0.03125],
        [0.75, 0.0625, 0.046875, 0.03515625, 0.0263671875, 0.0791015625],
        [0.8333333333, 0.0277777778, 0.0231481481, 0.0192901235, 0.0160751029, 0.0803755123],
        [0.875, 0.015625, 0.013671875, 0.0119628906, 0.0104675293, 0.0732727051],
    ]

    results = []
    overall_success = True
    
    v_counts = {state: [0] * 6 for state in states}
    for cycle in cycles:
        cycle_counter = Counter(cycle)
        for x_state in states:
            count = cycle_counter.get(x_state, 0)
            if count >= 5: v_counts[x_state][5] += 1
            else: v_counts[x_state][count] += 1
    
    for x_state in states:
        v_xk = v_counts[x_state]
        pi = pi_kx[abs(x_state) - 1]
        chi_sq = 0.0
        for k in range(6):
            numerator = (v_xk[k] - J * pi[k])**2
            denominator = J * pi[k]
            if denominator == 0: continue
            chi_sq += numerator / denominator
            
        p_value = ss.gammaincc(5.0 / 2.0, chi_sq / 2.0)
        if p_value < 0.01: overall_success = False
        results.append({'state': x_state, 'p_value': p_value, 'pass': (p_value >= 0.01)})

    return overall_success, results

# --- Cell 16: Random Excursions Variant Test ---
def random_excursions_variant_test(input_str):
    n = len(input_str)
    if n < 1000000:
        return [False, 0, None] # Not enough data

    x_np = np.array(list(input_str), dtype=np.int8) * 2 - 1
    s = np.cumsum(x_np)
    s_prime = np.concatenate(([0], s))
    J = np.count_nonzero(s_prime == 0)

    if J < 10:
        return [False, J, None] # Not enough cycles

    states = list(range(-9, 0)) + list(range(1, 10))
    results = []
    overall_success = True
    
    state_counts = Counter(s_prime)
    
    for x_state in states:
        count_x = state_counts.get(x_state, 0)
        numerator = abs(count_x - J)
        denominator = math.sqrt(2.0 * J * (4.0 * abs(x_state) - 2.0))
        
        if denominator == 0:
            p_value = 0.0
        else:
            p_value = ss.erfc(numerator / denominator)
        
        if p_value < 0.01:
            overall_success = False
        
        results.append({'state': x_state, 'count': count_x, 'p_value': p_value, 'pass': (p_value >= 0.01)})
        
    return overall_success, J, results

# ===================================================
# MAIN EXECUTION BLOCK — DIRECT .BIN TESTING
# ===================================================
if __name__ == "__main__":

    BIN_FILE = "tcn_rng_output (2).bin"
    BITS_TO_TEST = 1_280_000   # >= 1e6 required for full NIST

    print(f"Reading binary data from '{BIN_FILE}'...")

    try:
        with open(BIN_FILE, "rb") as f:
            byte_data = f.read()

        byte_array = np.frombuffer(byte_data, dtype=np.uint8)
        bit_array = np.unpackbits(byte_array)

        if len(bit_array) < BITS_TO_TEST:
            raise ValueError(
                f"Not enough bits: need {BITS_TO_TEST}, got {len(bit_array)}"
            )

        bit_array = bit_array[:BITS_TO_TEST]
        binary_data = "".join(bit_array.astype(str))
        data_length = len(binary_data)

        print(f"Loaded {data_length} bits successfully.")

    except Exception as e:
        print("ERROR reading .bin file:", e)
        sys.exit(1)

    print("\nRunning all 15 NIST Statistical Tests...")
    print("=" * 70)
    print(f"{'Test Name':<30} | {'P-Value(s)':<25} | {'Result'}")
    print("-" * 70)

    all_tests_passed = True

    def report(name, passed, pvals):
        print(f"{name:<30} | {pvals:<25} | {'PASS' if passed else 'FAIL'}")
        return passed

    # 1. Frequency
    r = frequency_test(binary_data, data_length)
    report("Frequency", r[4], f"{r[3]:.6f}")

    # 2. Block Frequency
    r = block_frequency_test(binary_data, data_length)
    report("Block Frequency", r[2], f"{r[1]:.6f}")

    # 3. Runs
    r = runs_test(binary_data, data_length)
    report("Runs", r[5], f"{r[4]:.6f}")

    # 4. Longest Run of Ones
    r = longest_run_test(binary_data)
    report("Longest Run", r[2], f"{r[1]:.6f}")

    # 5. Rank
    r = binary_matrix_rank_test(binary_data, data_length)
    report("Binary Matrix Rank", r[2], f"{r[1]:.6f}")

    # 6. Spectral (DFT)
    r = spectral_test(binary_data, data_length)
    report("Spectral (DFT)", r[4], f"{r[3]:.6f}")

    # 7. Non-overlapping Template
    r = non_overlapping_template_test(binary_data)
    report("Non-overlap Template", r[4], f"{r[3]:.6f}")

    # 8. Overlapping Template
    r = overlapping_template_test(binary_data)
    report("Overlapping Template", r[2], f"{r[1]:.6f}")

    # 9. Universal
    r = universal_test(binary_data)
    report("Universal", r[2], f"{r[1]:.6f}")

    # 10. Linear Complexity
    r = linear_complexity_test(binary_data)
    report("Linear Complexity", r[2], f"{r[1]:.6f}")

    # 11. Serial
    r = serial_test(binary_data, 10)
    report("Serial", r[4], f"{r[2]:.6f}, {r[3]:.6f}")

    # 12. Approximate Entropy
    r = approximate_entropy_test(binary_data)
    report("Approximate Entropy", r[3], f"{r[2]:.6f}")

    # 13. Cumulative Sums
    r = cumulative_sums_test(binary_data)
    report("Cumulative Sums", r[2], f"{r[0]:.6f}, {r[1]:.6f}")

    # 14. Random Excursions
    passed, res = random_excursions_test(binary_data)
    if res is None:
        report("Random Excursions", False, "Insufficient cycles")
    else:
        min_p = min(x["p_value"] for x in res)
        report("Random Excursions", passed, f"{min_p:.6f}")

    # 15. Random Excursions Variant
    passed, J, res = random_excursions_variant_test(binary_data)
    if res is None:
        report("Random Excursions Var", False, "Insufficient cycles")
    else:
        min_p = min(x["p_value"] for x in res)
        report("Random Excursions Var", passed, f"{min_p:.6f}")

    print("=" * 70)


Successfully imported gf2matrix.py
Reading binary data from 'tcn_rng_output (2).bin'...
ERROR reading .bin file: [Errno 2] No such file or directory: 'tcn_rng_output (2).bin'


AttributeError: 'tuple' object has no attribute 'tb_frame'